# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c21_surrogate_model_training  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-19
### Properties script
**Goal:** To train a surrogate model on a CSV dataset resulting from structural analyses done in grasshopper, this surrogate model should be able to accurately tell if a beam in the structure is structurally failing or not   
**Inputs:**
*   CSV with structural properties, from grasshopper

**Outputs:**
*   A surrogate model????

# PARAMETERS

In [ ]:
import pandas as pd
import numpy as np
import joblib
import config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error

CSV_FILE = 'MO_260317_9999_14_dataset_results.csv'  # De naam van het CSV-bestand dat je wilt gebruiken (staat in GH_DATA_PATH)

# Een Multi-Layer Perceptron (MLP). 
# hidden_layer_sizes=(64, 64) betekent: 2 verborgen lagen met elk 64 'neuronen'.
# max_iter=1000 betekent: Het netwerk mag 1000 keer proberen de foutmarge te verkleinen.
surrogate_model = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

In [ ]:
# BENAMING VAN BESTAND EN NETWERK ARCHITECTUUR
schone_naam = CSV_FILE.replace('.csv', '')
delen = schone_naam.split('_')

OPTIMIZATION_TYPE   = delen[0]                      # Pakt het 1e deel: 'SO'
DATE        = delen[1]
ITERATION           = delen[2]      # '260317_9999'

lagen_tuple = surrogate_model.hidden_layer_sizes
NETWORK_ARCH = "x".join(map(str, lagen_tuple))

MAX_ITER = surrogate_model.max_iter          # Haalt '1000' op
ACTIVATIE = surrogate_model.activation       # Haalt 'relu' op

prefix_sm = f"{OPTIMIZATION_TYPE}_{DATE}_{ITERATION}_arch{NETWORK_ARCH}_iter{MAX_ITER}"
with open(config.SM_EXPORT_PATH / f'prefix_sm.txt', 'w') as f:
    f.write(prefix_sm)

# IMPORTING DATA

In [ ]:
print("1. Dataset inladen...")
df = pd.read_csv(config.GH_DATA_PATH / CSV_FILE)
print(f"✅ Dataset '{CSV_FILE}' ingeladen.")

# INPUTS (X): Alle kolommen behalve de allereerste (index 0) en de allerlaatste (index -1)
INPUT_COLS = df.columns[1:-1].tolist()

# TARGETS (Y): Uitsluitend de allerlaatste kolom
TARGET_COLS = [df.columns[-1]]

print("\n--- Kolom Verdeling ---")
print(f"X (Inputs, {len(INPUT_COLS)} kolommen): {INPUT_COLS[:5]} ... (eerste 5 getoond)")
print(f"y (Target, 1 kolom): {TARGET_COLS}")

print(f"\nTotaal aantal samples in dataset CSV: {len(df)}")
print("\nVoorbeeld output (eerste 5 regels van dataset):")
print(df.head(5))

## Import Edge Index

In [ ]:
import config
import json

# In je Surrogate Model notebook laad je het straks simpelweg zo in:
with open('edge_index.json', 'r') as f:
    edge_index = json.load(f)

print("\nGegenereerde edge_index (eerste 5 verbindingen):")
print(f"Bron-nodes (V1): {edge_index[0][:5]}")
print(f"Doel-nodes (V2): {edge_index[1][:5]}")

# MODEL TRAINING

In [ ]:
import config

# Verwijder lege/irrelevante kolommen (bijv. de lege 'img' kolom van Colibri)
if 'img' in df.columns:
    df = df.drop(columns=['img'])

# Drop eventuele rijen met lege waarden (NaN)
df = df.dropna()

print(f"Dataset ingeladen met {len(df)} samples.")

# X EN Y SPLITSEN (Features en Targets)
X = df[INPUT_COLS]
y = df[TARGET_COLS]

# Splits de data: 80% trainen, 20% testen (evalueren hoe goed de AI het doet)
# random_state zorgt ervoor dat je elke keer dezelfde splitsing krijgt (reproduceerbaarheid!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Data gesplitst: {len(X_train)} trainingsamples en {len(X_test)} testsamples.")

# NORMALISATIE (Min-Max Scaling naar domein 0 - 1)
print("3. Data normaliseren...")

# Maak twee aparte 'meetlatten' aan: één voor de inputs, één voor de targets
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# BELANGRIJK: De scalers "leren" (fit) de minimum/maximum waarden
# UITSLUITEND van de trainingsdata, om spieken (data leakage) te voorkomen.
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

# Transformeer de test-data met de meetlat die zojuist is gemaakt
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

# Converteer terug naar DataFrames voor de overzichtelijkheid
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=INPUT_COLS)
y_train_scaled_df = pd.DataFrame(y_train_scaled, columns=TARGET_COLS)

print("\nVoorbeeld van de genormaliseerde Training Data (X):")
print(X_train_scaled_df.head(3))

# Nu is je data klaar om een neuraal netwerk (bijv met PyTorch/Keras) of een
# Random Forest model in te gooien!
# De variabelen die je nu aan je model voedt zijn: X_train_scaled en y_train_scaled

# MODEL TESTING

In [ ]:
import config

# ==========================================
# 2. HET NEURALE NETWERK BOUWEN (Surrogate Model)
# ==========================================
print("\n⚙️ Neuraal Netwerk aan het trainen...")

# Geef de genormaliseerde trainingsdata aan het model om te 'leren'
surrogate_model.fit(X_train_scaled, y_train_scaled)
print("✅ Training voltooid!")

# ==========================================
# 3. HET MODEL TESTEN (Evaluatie op ongeziene data)
# ==========================================
# Laat het model nu gokken (voorspellen) op de X_test data, die hij nog nooit gezien heeft!
y_pred_scaled = surrogate_model.predict(X_test_scaled)

# Als het model een platte lijst (1D) teruggeeft, dwingen we er een 2D-tabel (Matrix) van af.
# Dit voorkomt de ValueError bij de inverse_transform.
if len(y_pred_scaled.shape) == 1:
    y_pred_scaled = y_pred_scaled.reshape(-1, 1)

# Bereken hoe goed de gok is vergeleken met de échte antwoorden (y_test_scaled)
r2 = r2_score(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)

print("\n📊 --- RESULTATEN VAN HET MODEL ---")
print(f"R2 Score (Determinatiecoëfficiënt): {r2:.4f} (1.0 is perfect)")
print(f"Mean Squared Error (Foutmarge):   {mse:.4f} (0.0 is perfect)")

# ==========================================
# 4. TERUGREKENEN NAAR DE ECHTE WERELD (Inferentie-check)
# ==========================================
# Hier gebruiken we de scaler om de genormaliseerde (0-1) voorspellingen
# weer terug te transformeren naar fysieke millimeters en percentages!
y_pred_fysiek = scaler_y.inverse_transform(y_pred_scaled)

# Let op: als y_test_scaled 1D is geworden (vaak bij Pandas), fixen we die voor de zekerheid ook
if len(y_test_scaled.shape) == 1:
    y_test_scaled = y_test_scaled.reshape(-1, 1)
y_test_fysiek = scaler_y.inverse_transform(y_test_scaled)

# Laat een willekeurige constructie-iteratie zien (bijv. iteratie nummer 5 uit de test-set)
voorbeeld_idx = 5
print(f"\n🔍 --- VOORBEELD ITERATIE (Index {voorbeeld_idx}) ---")

# We loopen dynamisch door de TARGET_COLS heen, zodat het niet uitmaakt of je er 1 of 5 hebt!
for i, target_name in enumerate(TARGET_COLS):
    print(f"Werkelijke {target_name}: \t{y_test_fysiek[voorbeeld_idx][i]:.3f}")
    print(f"AI Voorspelt {target_name}: \t{y_pred_fysiek[voorbeeld_idx][i]:.3f}")
    print("-" * 40)

# EXPORT

In [ ]:
# We slaan de meetlatten op als .pkl bestanden.
# Deze heb je later nodig om de voorspellingen van je AI weer om te rekenen naar echte millimeters en kilo's!
joblib.dump(scaler_X, config.SM_EXPORT_PATH / f'scaler_X_.pkl')
joblib.dump(scaler_y, config.SM_EXPORT_PATH / f'scaler_y_{prefix_sm}.pkl')

print(f"\n✅ Scalers succesvol opgeslagen onder {config.SM_EXPORT_PATH}")

# Exporteer het getrainde netwerk zodat we het later in de optimalisatie-loop kunnen inladen!
joblib.dump(surrogate_model, config.SM_EXPORT_PATH / f'surrogate_model_{prefix_sm}.pkl')

print(f"\n💾 Model succesvol opgeslagen als onder {config.SM_EXPORT_PATH}")